# Transforms

In linear algebra, transform (or <wiki:transformation_matrix>) is used to specify the geometrical relationship between two <wiki:cartesian_coordinate_system>. There are many use cases:
- Co-registration between {term}`CT` and {term}`MRI> volumes
- Specify the {term}`3D` location of an object using tracking systems,
- Tool calibration, e.g. specifying the location of the tip of surgical drill with respect to the tracking sensor, etc.

That is, in an {term}`IGI` surgical workflow, there are always multiple **rigid bodies**, i.e. <wiki:cartesian_coordinate_system> present:
- Target anatomy (spine),
- Surgical tool (needle),
- Live image or video {term}`US`) and
- Geometrical model or representation of anatomy (e.g. surface model derived from {term}`CT`)

We must keep track of them all the time by understanding how each of these <wiki:cartesian_coordinate_system> are related with each other.

(ref_transform_translation)=
## Translation

The simplest transform is moving a object from one location to another, i.e. a [translation](https://en.wikipedia.org/wiki/Translation_(geometry)).

Let the initial position of an object be $p$, and let $v$ be a *translation vector*, then the new position of the object $q$ after translation by $v$ is:

$$
q = p + v
$$

If both $p$ and $v$ are multi-dimensional, then this is equivalant to vector addition. In {term}`3D`, we have
$$
\begin{align}
q_x & = & q_x + v_x\\
q_y & = & q_y + v_y\\
q_z & = & q_z + v_z
\end{align}
$$




In Python, we can simply use the standard addition operation:

In [8]:
import math
import numpy as np 
import numpy.matlib

p = np.array([ 1, 0, 0 ])      # This is a ROW vector
p = p.reshape(-1, 1)           # this is a COLUMN vector
p = np.array([ [0],[1],[0] ])  # alternative

v = np.array( [4, 5, 6] ).reshape(-1,1)
q = p + v

print("The initial position is:\n", p, 
"\nafter translation by:\n", v,
"\nis now located at:\n", q)

The initial position is:
 [[0]
 [1]
 [0]] 
after translation by:
 [[4]
 [5]
 [6]] 
is now located at:
 [[4]
 [6]
 [6]]


For visualization and other implementations, consult the following [tutorial](#ref_tutorial_translation).

(ref_transform_rotation)=
## Rotation

In {term}`3D`, rotation is specified as $3 \times 3$ matrix $R$, and using colomn vector, a point $p=(x,y,z)$ is rotated by $R$ into $q=(x',y',z')$ as:


$$
\begin{align}
A & = & \begin{bmatrix}
a & b & c\\
d & e & f\\
g & h & i
\end{bmatrix} \\
q & = & A p\\
& = & \begin{bmatrix}
a & b & c\\
d & e & f\\
g & h & i
\end{bmatrix} 
\begin{bmatrix}
x\\
y\\
z
\end{bmatrix} =

\begin{bmatrix}
x'\\
y'\\
z'
\end{bmatrix} \\
x' & = & a x + b y + c z\\
y' & = & d x + e y + f z \\
z' & = & g x + h y + i z
\end{align}
$$

### Rigid Rotation

We are particularly interested in rigid rotation, that that simply rotates an object but do not change its geometry (length, volumen, etc.,). For a $3 \times 3$ matrix to be a rigid rotation, the following properties must be satisfied:

1. Each row and each column is a <wiki:unit_vector>,
1. The <wiki:dot_product> between each row with every other is $0$.

In other word, a rigid rotation is akin to set up an **orthonormal axes**: each axis has a unit length one, and each axis is perpendicular with each other.

One consequence of these two properties is:
1. The <wiki:determinant> of a rigid rotation is $1$.

Let's look at some examples.

#### Rotation about z-axis

Perhaps the most familiar rotation is in fact the rotation about the $z$-axis. Most of us are used to perform or study rotation in {term}`2D`, i.e. the x-y plane, which is in fact equivalent to rotation about the $z$-axis.

The rotation is performed about an axis, called the axis of rotation, by an angle $\theta$. Using the {term}`RHR`, and when **looking from the *positive* towards the *negative* of the axis of rotation**, the {term}`CCW` rotation is the *positive* angle. 

That is, if the right-thumb points to the positive direction of the axis of rotation, then the curve of the fingers is the positive rotation.

It should be noted that a point on the axis of rotation is [invariant](https://en.wikipedia.org/wiki/Invariant_(mathematics)) under rotation.

Using the {term}`RHR`, rotation about $z$-axis is thus defined as:

$$
R_z(\theta) = \begin{bmatrix}
\cos{\theta} & -\sin{\theta} & 0\\
\sin{\theta} & \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix}
$$

Let's visualize this: suppose a point $p=(1,0,0)$ and $q=(0,1,0)$ is located on the $x$- and $y$-axis, respective, and after rotation, we have

$$
\begin{align}
p' & = & \begin{bmatrix}
\cos{\theta} & -\sin{\theta} & 0\\
\sin{\theta} & \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix} 
\begin{bmatrix}
1\\ 
0 \\
0 
\end{bmatrix} = \begin{bmatrix} \cos{\theta} \\ \sin{\theta} \\ 0 \end{bmatrix} \\
q' & = & \begin{bmatrix}
\cos{\theta} & -\sin{\theta} & 0\\
\sin{\theta} & \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix} 
\begin{bmatrix}
0\\ 
1 \\
0 
\end{bmatrix} = \begin{bmatrix} -\sin{\theta} \\ \cos{\theta} \\ 0 \end{bmatrix} 
\end{align}
$$

```{figure} ./../images/002/fig_rotz.png
:label: fig_rotz
:alt: Rotation about z-axis
:align: center
:width: 80%

Rotation about $z$-axis visualized.
```

And for any point $r=(0,0,z)$ on the $z$-axis, rotation about the $z$-axis does not move $r$:

$$
r'  =  \begin{bmatrix}
\cos{\theta} & -\sin{\theta} & 0\\
\sin{\theta} & \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix} 
\begin{bmatrix}
0\\ 
0 \\
z 
\end{bmatrix} = \begin{bmatrix} 0 \\ 0 \\ z \end{bmatrix} = r
$$

It can be implemented in Python as:

In [ ]:
import numpy as np
import math

def rotZ3x3( angle_rad ):
    # return a 3x3 rotation about z-axis, where angle is specified in radian

    c = math.cos( angle_rad )
    s = math.sin( angle_rad )
    # a 3x3 identity matrix
    R = np.identity(3)

    # specify the elements of the rotation
    R[0,0] =  c
    R[0,1] = -s
    R[1,0] =  s
    R[1,1] =  c
    return R

A simple test:

In [28]:
p = np.array([1,0,0]).reshape(-1,1)

# rotation by 90 degree
print( np.around( np.matmul( rotZ3x3( math.pi/2 ) , p) ) )
# print( rotZ3x3( math.pi/2 )@p )  # short cut, the @ sign is matrix multiplication

[[0.]
 [1.]
 [0.]]


#### Inverse Rotation

What happens if, instead rotation by $\theta$, one rotates by $-\theta$ instead?


$$
\begin{align}
R_z(\theta) & = &\begin{bmatrix}
\cos{\theta} & -\sin{\theta} & 0\\
\sin{\theta} & \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix}\\
R_z(-\theta) & = & \begin{bmatrix}
\cos{-\theta} & -\sin{-\theta} & 0\\
\sin{-\theta} & \cos{-\theta} & 0\\
0 & 0 & 1
\end{bmatrix}\\
& = & \begin{bmatrix}
\cos{\theta} & \sin{\theta} & 0\\
-\sin{\theta} & \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix}\\
\end{align}
$$

Suppose you rotate an object by an angle $\theta$, and immediately rotate it by an angle $-\theta$: nothing changes. Is it true numerically?

$$
R_z(-\theta) \cdot R_z(\theta) =  \begin{bmatrix}
\cos{\theta} & \sin{\theta} & 0\\
-\sin{\theta} & \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix} \cdot \begin{bmatrix}
\cos{\theta} & -\sin{\theta} & 0\\
\sin{\theta} & \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix}
$$


$$
\begin{align}
R_z(-\theta) \cdot R_z(\theta) & = & \begin{bmatrix}
\cos{\theta} & \sin{\theta} & 0\\
-\sin{\theta} & \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix} \cdot \begin{bmatrix}
\cos{\theta} & -\sin{\theta} & 0\\
\sin{\theta} & \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix} \\
& = & 
\begin{bmatrix}
\cos{\theta} \cos{\theta} + \sin{\theta} \sin{\theta} & -\cos{\theta} \sin{\theta} + \cos{\theta} \sin{\theta} & 0\\
-\cos{\theta} \sin{\theta} + \cos{\theta} \sin{\theta} & \sin{\theta} \sin{\theta} + \cos{\theta} \cos{\theta} & 0\\
0 & 0 & 1
\end{bmatrix}\\
& = & 
\begin{bmatrix}
\cos{\theta}^2 + \sin{\theta}^2 & 0 & 0\\
0 & \sin{\theta}^2 + \cos{\theta}^2 & 0\\
0 & 0 & 1
\end{bmatrix}\\
& = & 
\begin{bmatrix}
1 & 0 & 0\\
0 & 1 & 0\\
0 & 0 & 1
\end{bmatrix}
\end{align}
$$

You should also notice that $R_z(-\theta)=R_z(\theta)^T$, the transpose.

In [ ]:
# rotation by pi/4 followed by rotation by -pi/4
print( np.around( np.matmul( rotZ3x3( -math.pi/4 ) , rotZ3x3( math.pi/4)) ) )

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


#### Rotation about $x$-axis

To visualize how to perform rotation about the $x$-axis, position yourself at $+x$ direction looking towards the origin.
```{figure} ./../images/002/fig_rotx.png
:label: fig_rotx
:alt: Rotation about x-axis
:align: center
:width: 80%

Rotation about $x$-axis visualized.
```

Again, a point on the $x$-axis remains [invariant](https://en.wikipedia.org/wiki/Invariant_(mathematics)) under rotation about the $x$-axis.

Using this visualization, one can infer that the $3\times 3$ rotation about the $x$-axis by an angle $\theta$ is:

$$
R_x(\theta)  =  \begin{bmatrix}
1 & 0 & 0\\
0 & \cos{\theta} & -\sin{\theta}\\
0 & \sin{\theta} & \cos{\theta} 
\end{bmatrix} 
$$

In [ ]:
def rotX3x3( angle_rad ):
    # return a 3x3 rotation about z-axis, where angle is specified in radian

    c = math.cos( angle_rad )
    s = math.sin( angle_rad )
    # a 3x3 identity matrix
    R = np.identity(3)

    # specify the elements of the rotation
    R[1,1] =  c
    R[1,2] = -s
    R[2,1] =  s
    R[2,2] =  c
    return R

[[ 1.  0.  0.]
 [ 0.  0. -1.]
 [ 0.  1.  0.]]


#### Rotation about $y$-axis

To visualize how to perform rotation about the $y$-axis, position yourself at $+y$ direction looking towards the origin.
```{figure} ./../images/002/fig_roty.png
:label: fig_roty
:alt: Rotation about y-axis
:align: center
:width: 80%

Rotation about $y$-axis visualized.
```

Again, a point on the $y$-axis remains [invariant](https://en.wikipedia.org/wiki/Invariant_(mathematics)) under rotation about the $y$-axis.

Using this visualization, one can infer that the $3\times 3$ rotation about the $y$-axis by an angle $\theta$ is:

$$
R_x(\theta)  =  \begin{bmatrix}
\cos{\theta} & 0 & \sin{\theta}\\
0 & 1 & 0\\
-\sin{\theta} & 0 & \cos{\theta} 
\end{bmatrix} 
$$

In [7]:
def rotY3x3( angle_rad ):
    # return a 3x3 rotation about z-axis, where angle is specified in radian

    c = math.cos( angle_rad )
    s = math.sin( angle_rad )
    # a 3x3 identity matrix
    R = np.identity(3)

    # specify the elements of the rotation
    R[0,0] =  c
    R[0,2] =  s
    R[2,0] = -s
    R[2,2] =  c
    return R

### Successive Rotations

Suppose you are interested in rotating an object $p$ about the $x$-axis by an angle $\alpha$, and further by rotating the rotated object about the $y$-axis by an angle $\beta$, how would this be achieved?

Let $p'$ be the rotated object after the rotation about the $x$-axis by an angle $\alpha$:

$$
p'_{3\times 1} = R_x(\alpha)_{3\times 3} \cdot p_{3\times 1}
$$

and $p''$ be the rotated object after the rotation about the $y$-axis by an angle $\beta$:

$$
p''_{3 \times 1} = R_y(\beta)_{3 \times 3} \cdot p'_{3\times 1}
$$

Thus

$$
\begin{align}
p''_{3 \times 1} & = & R_y(\beta)_{3 \times 3} \cdot p'_{3\times 1}\\
& = & R_y(\beta)_{3 \times 3} \cdot R_x(\alpha)_{3\times 3} \cdot p_{3\times 1}
\end{align}
$$


Because matrix multiplication is [associative](https://en.wikipedia.org/wiki/Associative_property), we can indeed define a combined rotation matrix:

$$
R_{3 \times 3} =  R_y(\beta)_{3 \times 3} \cdot R_x(\alpha)_{3\times 3}
$$

and

$$
p''_{3 \times 1}=R_{3 \times 3} \cdot p_{3\times 1}
$$

That is, *instead of thinking about successive rotations as a series of rotations, **it is often more intuitive to think of it as just one rotation about an axis of rotation instead***.